# LeetCode 1957 — Delete Characters to Make Fancy String bằng Python

**Link đề:** https://leetcode.com/problems/delete-characters-to-make-fancy-string/
**Độ khó:** Easy · **Chủ đề:** String

```python
class Solution:
    def makeFancyString(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu yêu cầu "**không có 3 ký tự liên tiếp giống nhau**" và vì sao cách xoá tối thiểu là duy nhất.
2. Thấy nhược điểm của cách "xoá dần trên chuỗi" (`O(n²)`) và vì sao cộng chuỗi trong vòng lặp có thể chậm.
3. Thành thạo kỹ thuật **xây kết quả bằng list rồi `"".join`**, kiểm tra 2 phần tử cuối `res[-1]`, `res[-2]`.
4. Biết thêm 3 cách: đếm độ dài run, `itertools.groupby`, biểu thức chính quy `re.sub`.
5. Trace, chứng minh tính tối thiểu, Time/Space, lỗi hay gặp và biến thể (giữ tối đa `k` ký tự).

**Cách học:** đọc markdown → đoán output của cell code kế tiếp → chạy → so sánh. Các cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — chuỗi "fancy"

Một chuỗi là **fancy** nếu **không có ba ký tự liên tiếp bằng nhau**. Cho chuỗi `s`, hãy **xoá ít ký tự nhất** để `s` thành fancy và trả về chuỗi kết quả (đề đảm bảo kết quả là **duy nhất**).

| `s` | Output | Giải thích |
|---|---|---|
| `"leeetcode"` | `"leetcode"` | nhóm `eee` xoá 1 chữ `e` |
| `"aaabaaaa"` | `"aabaa"` | nhóm `aaa` xoá 1, nhóm `aaaa` xoá 2 |
| `"aab"` | `"aab"` | đã fancy |

**Constraints:** `1 <= s.length <= 10^5`, `s` chỉ gồm chữ thường.

Câu hỏi: với một nhóm `L` chữ giống nhau liên tiếp, giữ lại bao nhiêu chữ? → giữ `min(L, 2)`; xoá `L - 2` nếu `L > 2`.

In [1]:
s = "aaabaaaa"
print("Có 'aaa' trong s?", "aaa" in s)
print("Kết quả mong muốn:", "aabaa", "| có 'aaa'?", "aaa" in "aabaa")

Có 'aaa' trong s? True
Kết quả mong muốn: aabaa | có 'aaa'? False


## 2. Nền tảng Python cần dùng

### 2.1. Chuỗi là **bất biến** (immutable)

Không thể sửa một ký tự của `str` tại chỗ (`s[0] = 'x'` báo lỗi). Mọi thao tác "xoá" thực chất **tạo chuỗi mới** — tốn `O(n)` mỗi lần.

In [2]:
s = "leeetcode"
try:
    s[2] = "x"
except TypeError as e:
    print("TypeError:", e)
t = s[:2] + s[3:]          # "xoá" ký tự ở vị trí 2 = ghép hai lát cắt -> chuỗi MỚI
print(t, s)                # s vẫn nguyên

TypeError: 'str' object does not support item assignment
leetcode leeetcode


### 2.2. `list` làm "bộ đệm" và `"".join`

`list` thì **sửa được**: `res.append(ch)` thêm vào cuối trong `O(1)` (khấu hao). Khi xong, `"".join(res)` ghép tất cả thành chuỗi trong `O(n)`. Đây là cách chuẩn để **xây chuỗi từng ký tự** trong Python.

In [3]:
res = []
for ch in "abc":
    res.append(ch)
    print(res)
print("".join(res), "-".join(res))

['a']
['a', 'b']
['a', 'b', 'c']
abc a-b-c


### 2.3. Chỉ số âm `res[-1]`, `res[-2]`

`res[-1]` là phần tử **cuối**, `res[-2]` là phần tử **kế cuối**. Phải chắc `len(res) >= 2` trước khi đọc `res[-2]`, nếu không sẽ `IndexError`.

In [4]:
res = ["l", "e", "e"]
print(res[-1], res[-2], len(res))
try:
    print([][-1])
except IndexError as e:
    print("IndexError:", e)

e e 3
IndexError: list index out of range


### 2.4. `str.find` / `str.index` và `in` với chuỗi con

`s.find("eee")` trả vị trí đầu tiên của chuỗi con, hoặc `-1` nếu không có. `"eee" in s` chỉ trả `True/False`. Cả hai đều duyệt chuỗi (`O(n)` mỗi lần gọi).

In [5]:
s = "leeetcode"
print(s.find("eee"), s.find("zzz"), "eee" in s)

1 -1 True


## 3. Tính tay — nhóm theo run

`s = "aaabaaaa"` → run: `aaa | b | aaaa` → giữ `min(L, 2)` mỗi run: `aa | b | aa` → `"aabaa"`.

| Run | L | Giữ | Xoá |
|---|:---:|:---:|:---:|
| aaa | 3 | 2 | 1 |
| b | 1 | 1 | 0 |
| aaaa | 4 | 2 | 2 |

Tổng xoá `3` — không thể ít hơn (xem mục 10).

## 4. Cách 1 — Baseline: xoá dần trên chuỗi

Ý tưởng tự nhiên: *còn thấy 3 chữ giống nhau liên tiếp thì xoá một chữ*. Lặp tới khi hết.

In [6]:
def make_fancy_repeated_delete(s):
    i = 0
    while i + 2 < len(s):
        if s[i] == s[i + 1] == s[i + 2]:
            s = s[:i + 2] + s[i + 3:]     # xoá ký tự thứ ba -> tạo chuỗi mới O(n)
            # không tăng i: có thể vẫn còn bộ ba bắt đầu từ i
        else:
            i += 1
    return s

print(make_fancy_repeated_delete("leeetcode"))   # leetcode
print(make_fancy_repeated_delete("aaabaaaa"))    # aabaa
print(make_fancy_repeated_delete("aab"))         # aab

leetcode
aabaa
aab


### 4.1. Nút thắt của baseline

Mỗi lần xoá tạo chuỗi mới `O(n)`. Chuỗi `"a" * 100000` cần xoá `99998` lần → khoảng `n²/2 ≈ 5·10⁹` thao tác ký tự — quá chậm. Ta đếm thử số ký tự bị sao chép:

In [7]:
def copies_needed(n):
    # chuỗi "a"*n: mỗi lần xoá sao chép (độ dài hiện tại - 1) ký tự
    total, length = 0, n
    while length > 2:
        total += length - 1
        length -= 1
    return total

for n in [10, 1000, 100000]:
    print(n, "->", copies_needed(n), "ký tự bị sao chép")

10 -> 44 ký tự bị sao chép
1000 -> 499499 ký tự bị sao chép
100000 -> 4999949999 ký tự bị sao chép


## 5. Bước đột phá — quyết định **ngay khi đọc** từng ký tự

Thay vì xoá trên chuỗi gốc, ta **xây chuỗi kết quả** từ trái sang phải. Khi đọc `ch`, chỉ cần biết **hai ký tự cuối của kết quả**:
- nếu hai ký tự cuối đều bằng `ch` → thêm `ch` sẽ tạo bộ ba → **bỏ** `ch`;
- ngược lại → **thêm** `ch`.

Quyết định chỉ phụ thuộc 2 ký tự cuối → `O(1)` mỗi bước → `O(n)` tổng.

## 6. Lời giải chính — list + kiểm tra 2 ký tự cuối

**Đoạn 1:** `res = []`.

**Đoạn 2:** với mỗi `ch`: nếu `len(res) >= 2 and res[-1] == ch and res[-2] == ch` → bỏ qua; ngược lại `res.append(ch)`.

**Đoạn 3:** `return "".join(res)`.

In [8]:
# Đoạn 2 — thử điều kiện trên vài trạng thái
for res, ch in [(["e", "e"], "e"), (["l", "e"], "e"), (["e"], "e"), ([], "a")]:
    skip = len(res) >= 2 and res[-1] == ch and res[-2] == ch
    print(f"res={res}, ch={ch!r} -> {'BỎ' if skip else 'thêm'}")

res=['e', 'e'], ch='e' -> BỎ
res=['l', 'e'], ch='e' -> thêm
res=['e'], ch='e' -> thêm
res=[], ch='a' -> thêm


In [9]:
def make_fancy(s):
    res = []                                                    # Đoạn 1
    for ch in s:                                                # Đoạn 2
        if len(res) >= 2 and res[-1] == ch and res[-2] == ch:
            continue                                            # ký tự thứ 3 liên tiếp -> bỏ
        res.append(ch)
    return "".join(res)                                         # Đoạn 3

print(make_fancy("leeetcode"), make_fancy("aaabaaaa"), make_fancy("aab"))

leetcode aabaa aab


`continue` nhảy ngay sang vòng lặp tiếp theo, bỏ qua phần còn lại của thân vòng (ở đây là `append`).

## 7. TRACE từng bước

Ví dụ `s = "aaabaaaa"`.

| i | ch | res trước | 2 ký tự cuối | Quyết định | res sau |
|---:|:---:|---|---|---|---|
| 0 | a | `[]` | — | thêm | `a` |
| 1 | a | `a` | — (chỉ có 1) | thêm | `aa` |
| 2 | a | `aa` | `a a` | **bỏ** | `aa` |
| 3 | b | `aa` | `a a` ≠ b | thêm | `aab` |
| 4 | a | `aab` | `a b` | thêm | `aaba` |
| 5 | a | `aaba` | `b a` | thêm | `aabaa` |
| 6 | a | `aabaa` | `a a` | **bỏ** | `aabaa` |
| 7 | a | `aabaa` | `a a` | **bỏ** | `aabaa` |

In [10]:
def trace_make_fancy(s):
    res = []
    for i, ch in enumerate(s):
        before = "".join(res)
        last_two = "".join(res[-2:])          # res[-2:] an toàn kể cả khi res ngắn
        if len(res) >= 2 and res[-1] == ch and res[-2] == ch:
            print(f"  i={i} ch={ch} res={before!r:<9} cuối={last_two!r:<5} -> BỎ")
            continue
        res.append(ch)
        print(f"  i={i} ch={ch} res={before!r:<9} cuối={last_two!r:<5} -> thêm, res={''.join(res)!r}")
    return "".join(res)

print("Kết quả:", trace_make_fancy("aaabaaaa"))

  i=0 ch=a res=''        cuối=''    -> thêm, res='a'
  i=1 ch=a res='a'       cuối='a'   -> thêm, res='aa'
  i=2 ch=a res='aa'      cuối='aa'  -> BỎ
  i=3 ch=b res='aa'      cuối='aa'  -> thêm, res='aab'
  i=4 ch=a res='aab'     cuối='ab'  -> thêm, res='aaba'
  i=5 ch=a res='aaba'    cuối='ba'  -> thêm, res='aabaa'
  i=6 ch=a res='aabaa'   cuối='aa'  -> BỎ
  i=7 ch=a res='aabaa'   cuối='aa'  -> BỎ
Kết quả: aabaa


Chú ý: lát cắt `res[-2:]` **không** lỗi khi `res` ngắn hơn 2 (trả list ngắn hơn), khác với chỉ số `res[-2]`.

### 7.1. Nhánh đặc biệt: run dài và các run xen kẽ

In [11]:
print("Kết quả:", trace_make_fancy("bbbbbab"))

  i=0 ch=b res=''        cuối=''    -> thêm, res='b'
  i=1 ch=b res='b'       cuối='b'   -> thêm, res='bb'
  i=2 ch=b res='bb'      cuối='bb'  -> BỎ
  i=3 ch=b res='bb'      cuối='bb'  -> BỎ
  i=4 ch=b res='bb'      cuối='bb'  -> BỎ
  i=5 ch=a res='bb'      cuối='bb'  -> thêm, res='bba'
  i=6 ch=b res='bba'     cuối='ba'  -> thêm, res='bbab'
Kết quả: bbab


## 8. Các cách giải khác

### 8.1. Cách 3 — đếm độ dài run hiện tại

Giữ `count` = số ký tự giống `ch` liên tiếp **trong `s`** tính đến vị trí hiện tại. Chỉ thêm khi `count <= 2`. Không cần nhìn vào `res`.

In [12]:
def make_fancy_count(s):
    res = []
    count = 0
    for i, ch in enumerate(s):
        count = count + 1 if i > 0 and ch == s[i - 1] else 1   # biểu thức điều kiện (ternary)
        if count <= 2:
            res.append(ch)
    return "".join(res)

print(make_fancy_count("leeetcode"), make_fancy_count("aaabaaaa"), make_fancy_count("aab"))
print("ví dụ ternary:", "chẵn" if 4 % 2 == 0 else "lẻ")

leetcode aabaa aab
ví dụ ternary: chẵn


### 8.2. Cách 4 — `itertools.groupby`

Mỗi run `(ch, group)`, giữ `ch * min(len, 2)`.

In [13]:
from itertools import groupby

def make_fancy_groupby(s):
    return "".join(ch * min(2, len(list(g))) for ch, g in groupby(s))

print(make_fancy_groupby("leeetcode"), make_fancy_groupby("aaabaaaa"), make_fancy_groupby("aab"))
print("a" * 2, "a" * 1)   # nhân chuỗi với số = lặp chuỗi

leetcode aabaa aab
aa a


### 8.3. Cách 5 — biểu thức chính quy

Mẫu `(.)\1{2,}` nghĩa là: một ký tự bất kỳ (nhóm 1), theo sau bởi **ít nhất 2** lần lặp lại chính nó — tức một run dài ≥ 3. Thay cả run bằng `\1\1` (hai lần ký tự đó).

In [14]:
import re

def make_fancy_regex(s):
    return re.sub(r"(.)\1{2,}", r"\1\1", s)

print(make_fancy_regex("leeetcode"), make_fancy_regex("aaabaaaa"), make_fancy_regex("aab"))
print(re.findall(r"((.)\2{2,})", "aaabaaaa"))   # các run dài >= 3 được tìm thấy

leetcode aabaa aab
[('aaa', 'a'), ('aaaa', 'a')]


### 8.4. Vì sao không cộng chuỗi `res += ch`?

`res = res + ch` về lý thuyết tạo chuỗi mới mỗi lần (`O(n)`) → `O(n²)`. CPython có tối ưu đặc biệt khiến `+=` thường nhanh, nhưng đó là **chi tiết cài đặt**, không được đảm bảo (và ta còn cần đọc 2 ký tự cuối). Dùng list + `join` là thói quen an toàn.

In [15]:
def make_fancy_str_concat(s):
    res = ""
    for ch in s:
        if len(res) >= 2 and res[-1] == ch and res[-2] == ch:
            continue
        res += ch           # chạy đúng, nhưng độ phức tạp phụ thuộc tối ưu của trình thông dịch
    return res

print(make_fancy_str_concat("aaabaaaa"))

aabaa


## 9. Code nộp LeetCode (Python 3)

In [16]:
class Solution:
    def makeFancyString(self, s: str) -> str:
        res = []
        for ch in s:
            if len(res) >= 2 and res[-1] == ch and res[-2] == ch:
                continue
            res.append(ch)
        return "".join(res)


sol = Solution()
assert sol.makeFancyString("leeetcode") == "leetcode"
assert sol.makeFancyString("aaabaaaa") == "aabaa"
assert sol.makeFancyString("aab") == "aab"
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


## 10. Vì sao thuật toán đúng (và tối thiểu)?

**1. Kết quả là fancy.** Invariant: *`res` luôn fancy*. Ban đầu rỗng — đúng. Khi thêm `ch`, ta chỉ thêm nếu **không** phải cả hai ký tự cuối đều bằng `ch`, nên không tạo bộ ba mới ở cuối; phần trước đã fancy theo giả thiết.

**2. Số ký tự xoá là tối thiểu.** Xét một run gồm `L` ký tự `c` liên tiếp trong `s`. Sau khi xoá, các ký tự `c` còn lại của run này vẫn đứng **liền nhau** (giữa chúng chỉ có các ký tự cũng thuộc run, tức cũng là `c`). Chuỗi fancy chỉ chứa tối đa 2 chữ `c` liền nhau → phải xoá **ít nhất** `L - 2` ký tự của run (nếu `L > 2`). Thuật toán xoá **đúng** `max(0, L - 2)` cho mỗi run → đạt cận dưới.

Vì sao thuật toán xoá đúng như vậy? Trong một run, hai chữ đầu được thêm; từ chữ thứ ba trở đi, hai ký tự cuối của `res` là `c c` nên bị bỏ. Khi sang run mới (ký tự khác), ký tự cuối của `res` khác `ch` nên chắc chắn được thêm — **các run không ảnh hưởng nhau**.

**3. Duy nhất.** Mỗi run giữ đúng 2 (hoặc `L` nếu `L ≤ 2`) ký tự, mà các ký tự trong run giống hệt nhau → chuỗi kết quả duy nhất.

## 11. Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Xoá dần trên chuỗi | `O(n²)` | `O(n)` | quá chậm với `n = 10^5` |
| 2. List + 2 ký tự cuối (chính) | `O(n)` | `O(n)` cho kết quả | nên nộp |
| 3. Đếm run | `O(n)` | `O(n)` | không cần nhìn `res` |
| 4. `groupby` | `O(n)` | `O(n)` | ngắn gọn |
| 5. `re.sub` | `O(n)` | `O(n)` | một dòng, cần hiểu regex |
| (cộng chuỗi `+=`) | `O(n)`–`O(n²)` tuỳ trình thông dịch | `O(n)` | tránh |

Bộ nhớ `O(n)` là bắt buộc vì đầu ra chính là một chuỗi độ dài tới `n`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

sol = Solution()
check("đề 1", sol.makeFancyString("leeetcode"), "leetcode")
check("đề 2", sol.makeFancyString("aaabaaaa"), "aabaa")
check("đề 3", sol.makeFancyString("aab"), "aab")
check("một ký tự", sol.makeFancyString("z"), "z")
check("hai ký tự giống", sol.makeFancyString("zz"), "zz")
check("ba ký tự giống", sol.makeFancyString("zzz"), "zz")
check("run dài", sol.makeFancyString("a" * 10), "aa")
check("xen kẽ không xoá", sol.makeFancyString("ababab"), "ababab")
check("run ở cuối", sol.makeFancyString("abccccc"), "abcc")
check("nhiều run", sol.makeFancyString("aaabbbccc"), "aabbcc")

PASS đề 1: 'leetcode'
PASS đề 2: 'aabaa'
PASS đề 3: 'aab'
PASS một ký tự: 'z'
PASS hai ký tự giống: 'zz'
PASS ba ký tự giống: 'zz'
PASS run dài: 'aa'
PASS xen kẽ không xoá: 'ababab'
PASS run ở cuối: 'abcc'
PASS nhiều run: 'aabbcc'


In [18]:
# Đối chiếu các cách + kiểm tra tính chất trên dữ liệu ngẫu nhiên có seed
import random
random.seed(1957)
funcs = [make_fancy_repeated_delete, make_fancy, make_fancy_count, make_fancy_groupby,
         make_fancy_regex, make_fancy_str_concat, Solution().makeFancyString]

def is_fancy(t):
    return all(not (t[i] == t[i + 1] == t[i + 2]) for i in range(len(t) - 2))

def is_subsequence(t, s):
    it = iter(s)
    return all(ch in it for ch in t)     # mẹo: 'in' trên iterator tiêu thụ dần

for _ in range(500):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 30)))
    results = {f(s) for f in funcs}
    assert len(results) == 1, (s, results)
    out = results.pop()
    assert is_fancy(out) and is_subsequence(out, s)
print("500 chuỗi ngẫu nhiên: mọi cách khớp, kết quả fancy và là dãy con của s")
print("chuỗi 10^5 ký tự 'a' ->", len(sol.makeFancyString("a" * 100000)), "ký tự")

500 chuỗi ngẫu nhiên: mọi cách khớp, kết quả fancy và là dãy con của s
chuỗi 10^5 ký tự 'a' -> 2 ký tự


## 13. Sai lầm người mới hay mắc

### Lỗi 1 — Chỉ nhìn **một** ký tự cuối

```python
if res and res[-1] == ch:   # SAI: cấm cả 2 ký tự liên tiếp giống nhau
    continue
```
Điều kiện này giữ tối đa **1** ký tự mỗi run — xoá quá tay (đó là bài "không có 2 ký tự liên tiếp giống nhau").

In [19]:
def buggy_one_last(s):
    res = []
    for ch in s:
        if res and res[-1] == ch:
            continue
        res.append(ch)
    return "".join(res)

for t in ["leeetcode", "aab"]:
    print(repr(t), "buggy:", buggy_one_last(t), "| đúng:", make_fancy(t))

'leeetcode' buggy: letcode | đúng: leetcode
'aab' buggy: ab | đúng: aab


Bản lỗi biến `"aab"` (vốn đã fancy) thành `"ab"` — xoá thừa. Bộ ba cần **hai** ký tự cuối giống `ch`.

### Lỗi 2 — Đọc `res[-2]` khi `res` chỉ có 1 phần tử

In [20]:
def buggy_index_error(s):
    res = []
    for ch in s:
        if res and res[-1] == ch and res[-2] == ch:    # thiếu kiểm tra len(res) >= 2
            continue
        res.append(ch)
    return "".join(res)

try:
    buggy_index_error("aa")
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


Sửa: `len(res) >= 2 and ...` — nhờ **đánh giá ngắn mạch** của `and`, `res[-2]` chỉ được đọc khi đã chắc có ít nhất 2 phần tử.

### Lỗi 3 — Chỉ xoá **một** ký tự cho mỗi run

In [21]:
def buggy_delete_once(s):
    out, i = [], 0
    while i < len(s):
        j = i
        while j < len(s) and s[j] == s[i]:
            j += 1
        L = j - i
        out.append(s[i] * (L - 1 if L >= 3 else L))   # SAI: run dài 4 vẫn còn 3
        i = j
    return "".join(out)

print("buggy:", buggy_delete_once("aaaab"), "| đúng:", make_fancy("aaaab"))

buggy: aaab | đúng: aab


Run dài `L` phải giữ `min(L, 2)`, không phải `L - 1`.

### Lỗi 4 — Dùng `s.replace("aaa", "aa")` một lần

`replace` thay các lần xuất hiện **không chồng nhau** một lượt: `"aaaa".replace("aaa", "aa")` → `"aaa"` — vẫn còn bộ ba!

In [22]:
print("aaaa".replace("aaa", "aa"))      # 'aaa' — chưa fancy
print("aaaaa".replace("aaa", "aa"))     # 'aaaa'

aaa
aaaa


### Lỗi 5 — Cộng chuỗi trong vòng lặp ở ngôn ngữ khác

Thói quen `res = res + ch` sẽ thành `O(n²)` trong Java/JS cũ… và không được đảm bảo nhanh ở Python. Dùng list + `join` (hoặc `StringBuilder` trong Java).

## 14. Mẹo nhận dạng pattern

- "Không có `k` phần tử liên tiếp giống nhau" → nghĩ tới **run** và giữ `min(L, k-1)`.
- "Xây chuỗi kết quả từng ký tự với điều kiện dựa trên **phần cuối** của kết quả" → **list như stack** (`res[-1]`, `res[-2]`, `append`/`pop`). Cùng họ: 1047 (Remove All Adjacent Duplicates In String), 1544 (Make The String Great), 1209 (Remove All Adjacent Duplicates in String II).
- Thấy "xoá dần cho tới khi hết vi phạm" → cảnh giác `O(n²)`; hỏi "có quyết định được ngay khi đọc không?".
- Bài anh em: 1933 (cắt run theo `% 3`), 443 (String Compression).

## 15. Biến thể

### 15.1. Tổng quát: không có `k` ký tự liên tiếp giống nhau

In [23]:
def make_fancy_k(s, k):
    res = []
    count = 0
    for i, ch in enumerate(s):
        count = count + 1 if i > 0 and ch == s[i - 1] else 1
        if count < k:            # giữ tối đa k - 1 ký tự mỗi run
            res.append(ch)
    return "".join(res)

print(make_fancy_k("aaabaaaa", 3), make_fancy_k("aaabaaaa", 2), make_fancy_k("aaabaaaa", 4))

aabaa aba aaabaaa


### 15.2. Chỉ đếm số ký tự phải xoá

In [24]:
def deletions_needed(s):
    return sum(max(0, len(list(g)) - 2) for _, g in groupby(s))

print(deletions_needed("aaabaaaa"), deletions_needed("leeetcode"), deletions_needed("aab"))   # 3 1 0

3 1 0


### 15.3. LeetCode 1047 — xoá cặp ký tự kề nhau giống nhau (stack)

Khác bài này ở chỗ việc xoá có thể **lan truyền** (`"abba"` → `"aa"` → `""`), nên phải dùng stack thực sự với `pop`.

In [25]:
def remove_adjacent_duplicates(s):
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()             # gặp cặp -> xoá cả hai
        else:
            stack.append(ch)
    return "".join(stack)

print(remove_adjacent_duplicates("abbaca"), remove_adjacent_duplicates("abba"))   # 'ca' ''

ca 


## 16. Bài tập (TODO — tự làm, không chạy tự động)

**Bài 1 (dự đoán output):** `"xyz"[-2:]`, `[1][-2:]`, `"ab" * 0`, `min(5, 2)`.

**Bài 2 (tính tay):** lập bảng trace cho `s = "ccccbbbc"` và cho kết quả.

**Bài 3 (điền chỗ trống):** hoàn thiện lời giải chính.

In [26]:
def exercise_make_fancy(s):
    res = []
    for ch in s:
        # TODO 1: nếu res có >= 2 phần tử và 2 phần tử cuối đều bằng ch -> bỏ qua
        # TODO 2: ngược lại thêm ch vào res
        pass
    return "".join(res)

# assert exercise_make_fancy("leeetcode") == "leetcode"
# assert exercise_make_fancy("aaabaaaa") == "aabaa"

**Bài 4 (tự viết lại):** viết `exercise_make_fancy_count(s)` dùng biến `count` (độ dài run hiện tại), **không** đọc `res`.

In [27]:
def exercise_make_fancy_count(s):
    # TODO
    pass

# assert exercise_make_fancy_count("aaabaaaa") == "aabaa"
# assert exercise_make_fancy_count("z") == "z"

**Bài 5 (biến thể):** viết `exercise_deleted_indices(s)` trả list **chỉ số** (trong `s` gốc) của các ký tự bị xoá, tăng dần.

**Bài 6 (mini challenge):** viết `exercise_remove_k_adjacent(s, k)` (LeetCode 1209): lặp lại việc xoá **đúng `k`** ký tự giống nhau liền kề cho tới khi không còn; dùng stack chứa cặp `[ký_tự, số_lượng]`.

In [28]:
def exercise_deleted_indices(s):
    # TODO
    pass

def exercise_remove_k_adjacent(s, k):
    # TODO
    pass

# assert exercise_deleted_indices("aaabaaaa") == [2, 6, 7]
# assert exercise_deleted_indices("aab") == []
# assert exercise_remove_k_adjacent("deeedbbcccbdaa", 3) == "aa"
# assert exercise_remove_k_adjacent("abcd", 2) == "abcd"

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | lát cắt không lỗi khi ngắn | `[1][-2:]` → list 1 phần tử | `"ab" * 0` là chuỗi rỗng |
| 2 | chia run | `cccc`, `bbb`, `c` | giữ `cc`, `bb`, `c` |
| 3 | 2 ký tự cuối | `len(res) >= 2 and res[-1] == ch and res[-2] == ch` | `continue` / `append` |
| 4 | run trên `s` gốc | so `ch` với `s[i-1]` | `count = count + 1 if ... else 1`; thêm khi `count <= 2` |
| 5 | ký tự thứ 3+ của run | dùng `count` như bài 4 | khi `count > 2` thì `append(i)` vào list chỉ số |
| 6 | stack các cặp | đỉnh cùng ký tự thì tăng số đếm | đếm tới `k` thì `pop`; cuối cùng ghép `ch * cnt` |

## 18. Đáp án tham khảo — chỉ xem sau khi tự làm

**Bài 1:** `'yz'`, `[1]`, `''`, `2`.

**Bài 2:** `"ccccbbbc"` → run `cccc | bbb | c` → `"ccbbc"`.

In [29]:
print("xyz"[-2:], [1][-2:], repr("ab" * 0), min(5, 2))
print(make_fancy("ccccbbbc"))


def answer_make_fancy(s):
    res = []
    for ch in s:
        if len(res) >= 2 and res[-1] == ch and res[-2] == ch:
            continue
        res.append(ch)
    return "".join(res)


def answer_make_fancy_count(s):
    res, count = [], 0
    for i, ch in enumerate(s):
        count = count + 1 if i > 0 and ch == s[i - 1] else 1
        if count <= 2:
            res.append(ch)
    return "".join(res)


def answer_deleted_indices(s):
    deleted, count = [], 0
    for i, ch in enumerate(s):
        count = count + 1 if i > 0 and ch == s[i - 1] else 1
        if count > 2:
            deleted.append(i)
    return deleted


def answer_remove_k_adjacent(s, k):
    stack = []                           # mỗi phần tử: [ký tự, số lượng liền nhau]
    for ch in s:
        if stack and stack[-1][0] == ch:
            stack[-1][1] += 1
            if stack[-1][1] == k:
                stack.pop()
        else:
            stack.append([ch, 1])
    return "".join(ch * cnt for ch, cnt in stack)


assert answer_make_fancy("leeetcode") == "leetcode"
assert answer_make_fancy("aaabaaaa") == "aabaa"
assert answer_make_fancy_count("aaabaaaa") == "aabaa"
assert answer_make_fancy_count("z") == "z"
assert answer_deleted_indices("aaabaaaa") == [2, 6, 7]
assert answer_deleted_indices("aab") == []
assert answer_remove_k_adjacent("deeedbbcccbdaa", 3) == "aa"
assert answer_remove_k_adjacent("abcd", 2) == "abcd"
for _ in range(200):
    t = "".join(random.choice("abc") for _ in range(random.randint(1, 20)))
    kept = [ch for i, ch in enumerate(t) if i not in set(answer_deleted_indices(t))]
    assert "".join(kept) == make_fancy(t)
print("Đáp án tham khảo PASS toàn bộ test!")

yz [1] '' 2
ccbbc
Đáp án tham khảo PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
class Solution:
    def makeFancyString(self, s: str) -> str:
        res = []
        for ch in s:
            if len(res) >= 2 and res[-1] == res[-2] == ch:   # sẽ thành bộ ba -> bỏ
                continue
            res.append(ch)
        return "".join(res)
```

**Câu thần chú:** *mỗi run giữ tối đa 2 — nhìn 2 ký tự cuối của kết quả, xây bằng list rồi join.*

**Checklist trước khi submit:**
- [ ] Kiểm tra `len(res) >= 2` **trước** khi đọc `res[-2]`?
- [ ] Xây kết quả bằng list + `"".join`, không xoá dần trên chuỗi (`O(n²)`)?
- [ ] Run dài 4, 5, 10 đều còn đúng 2 ký tự?
- [ ] Không dùng `replace("aaa", "aa")` một lần?
- [ ] Test `"a" * 100000` chạy nhanh?